In [3]:
pip install bleak


   ------------------------------------ ---  9/10 [bleak]
   ---------------------------------------- 10/10 [bleak]

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install mediapipe==0.10.14

   ---------------------------------------- 0.0/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/50.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/50.8 MB 1.9 MB/s eta 0:00:27
    --------------------------------------- 1.0/50.8 MB 1.9 MB/s eta 0:00:26
   - -------------------------------------- 1.6/50.8 MB 2.1 MB/s eta 0:00:24
   - -------------------------------------- 2.1/50.8 MB 2.3 MB/s eta 0:00:22
   - -------------------------------------- 2.1/50.8 MB 2.3 MB/s eta 0:00:22
   - -------------------------------------- 2.4/50.8 MB 1.6 MB/s eta 0:00:30
   -- ------------------------------------- 3.1/50.8 MB 1.9 MB/s eta 0:00:26
   --- ------------------------------------ 3.9/50.8 MB 2.1 MB/s eta 0:00:22
   --- ------------------------------------ 4.7/50.8 MB 2.3 MB/s eta 0:00:20
   ---- ----------------------------------- 5.8/50.8 MB 2.6 MB/s eta 0:00:18
   ----- ---------------------------------- 6.6/50.8 MB 2.7 MB/s eta 0:00:17
   ----- ----

In [ ]:
import asyncio
import json
import os
import socket
import threading
import math
import time
import cv2
import mediapipe as mp
from bleak import BleakScanner
import nest_asyncio

nest_asyncio.apply()

DB_FILE = "registered_users.json"
RSSI_THRESHOLD = -60

def load_users():
    if os.path.exists(DB_FILE):
        with open(DB_FILE, "r") as f:
            return json.load(f)
    return {}

def save_users(users):
    with open(DB_FILE, "w") as f:
        json.dump(users, f, indent=4)

async def proximity_radar(conn):
    print(f"📡 Background Bluetooth Radar Started...")
    conn.setblocking(False)
    current_user_mac = None

    while True:
        try:
            data = conn.recv(1024)
            if not data: break
            if b"LOGOUT" in data:
                print("\n[Manual Logout] Resuming radar...\n")
                current_user_mac = None  
        except BlockingIOError:
            pass  
        except (ConnectionResetError, BrokenPipeError):
            break

        if current_user_mac is None:
            devices_dict = await BleakScanner.discover(timeout=3.0, return_adv=True)
            close_devices = []
            for mac_addr, (device, adv_data) in devices_dict.items():
                name = adv_data.local_name or device.name
                if name and str(name).strip() != "0" and adv_data.rssi > RSSI_THRESHOLD:
                    close_devices.append({"address": mac_addr, "name": str(name).strip(), "rssi": adv_data.rssi})
            
            if close_devices:
                closest = max(close_devices, key=lambda d: d["rssi"])
                users = load_users()
                if closest["address"] not in users:
                    users[closest["address"]] = {"username": closest['name']}
                    save_users(users)
                
                username = users[closest["address"]]["username"]
                print(f"✅ Logging in: {username}")
                try:
                    conn.send(bytes(f"LOGIN:{username}\n", "utf-8"))
                    current_user_mac = closest["address"]
                except: break
                    
        await asyncio.sleep(1)

def start_bluetooth_thread(conn):
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(proximity_radar(conn))

class CircularMenuController:
    def __init__(self):
        self.state = "HIDDEN"
        self.current_selection = None
        self.last_hand_time = time.time()
        self.DROP_TIMEOUT = 0.8 

    def _get_open_fingers(self, landmarks):
        tips, pips = [8, 12, 16, 20], [6, 10, 14, 18]
        open_count = 0
        for tip, pip in zip(tips, pips):
            if landmarks.landmark[tip].y < landmarks.landmark[pip].y:
                open_count += 1
        return open_count

    def _get_menu_slice(self, landmarks):
        # FIX: Invert DX calculation to handle the mirrored camera view
        # We subtract the wrist from the middle finger, then flip the sign
        dx = -(landmarks.landmark[9].x - landmarks.landmark[0].x)
        dy = landmarks.landmark[0].y - landmarks.landmark[9].y 
        
        angle = math.degrees(math.atan2(dy, dx))
        if angle < 0: angle += 360
            
        if 60 <= angle <= 120: return "Hint"       
        elif 120 < angle <= 240: return "Restart"  
        elif 240 < angle <= 360 or 0 <= angle < 60: return "Logout" 
        return self.current_selection

    def process_hand(self, landmarks):
        if landmarks is None:
            if self.state == "ACTIVE" and (time.time() - self.last_hand_time) > self.DROP_TIMEOUT:
                self.state = "HIDDEN"
                return "CANCEL"
            return None

        self.last_hand_time = time.time()
        open_f = self._get_open_fingers(landmarks)
        if self.state == "HIDDEN" and open_f >= 3:
            self.state = "ACTIVE"
            self.current_selection = self._get_menu_slice(landmarks)
            return f"OPEN_MENU:{self.current_selection}"
        elif self.state == "ACTIVE":
            selection = self._get_menu_slice(landmarks)
            if open_f <= 1: # Fist to select
                self.state = "HIDDEN"
                res = self.current_selection
                self.current_selection = None
                return f"SELECT:{res}"
            if selection != self.current_selection:
                self.current_selection = selection
                return f"HOVER:{selection}"
        return None

if __name__ == "__main__":
    mySocket = socket.socket()
    mySocket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    mySocket.bind(("localhost", 5000))
    mySocket.listen(1)
    print("⏳ Waiting for C#...")
    conn, addr = mySocket.accept()
    
    threading.Thread(target=start_bluetooth_thread, args=(conn,), daemon=True).start()

    mp_hands = mp.solutions.hands
    menu = CircularMenuController()
    cap = cv2.VideoCapture(1) # Change to 0 if laptop cam, 1 for phone
    
    with mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7, max_num_hands=1) as hands:
        while cap.isOpened():
            success, image = cap.read()
            if not success: break
            image = cv2.flip(image, 1) 
            results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            landmarks = results.multi_hand_landmarks[0] if results.multi_hand_landmarks else None
            command = menu.process_hand(landmarks)
            if command:
                try: conn.send(bytes(command + "\n", "utf-8"))
                except: break
            if results.multi_hand_landmarks:
                for hl in results.multi_hand_landmarks:
                    mp.solutions.drawing_utils.draw_landmarks(image, hl, mp_hands.HAND_CONNECTIONS)
            cv2.imshow('MediaPipe Tracker', image)
            if cv2.waitKey(5) & 0xFF == 27: break
    cap.release()
    cv2.destroyAllWindows()
    conn.close()
    mySocket.close()

⏳ Waiting for C#...
📡 Background Bluetooth Radar Started...


c:\Users\yahia\anaconda3\envs\HCIVEnv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✅ Logging in: Galaxy Buds Pro (6FB9) LE

[Manual Logout] Resuming radar...

✅ Logging in: Galaxy Buds Pro (6FB9) LE
